# IOAI — 2024 On Site Round Help Bobai — ⭐모범답안 (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/IOAI-official/IOAI-2024"
CLONE = "IOAI-2024"
SUBDIR = "On-Site-Round/Help_BOBAI"
WORKDIR = "On-Site-Round/Help_BOBAI/Solution"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

<img src="./figs/IOAI-Logo.png" alt="IOAI Logo" width="200" height="auto">

[IOAI 2024 (불가리아 부르가스), 온사이트 라운드](https://ioai-official.org/bulgaria-2024)

# Help BOBAI — 모범답안2 (공식 우수 풀이 기반, cosine-KNN 게이팅)

**출처**: IOAI 공식 *Best solutions* 아카이브의 `Best_solution_IOAI_2024_NLP.ipynb` 접근법.

**과제**: 동결된 5-way 분류기(파라미터 변경·신규 학습 금지)에 신규 클래스 2개(5,6)를 추가해 7-way 분류.
허용 연산은 인코딩의 **평균·거리**뿐.

**방법**: 학습 라벨을 `{기존<5 → 0, ==5 → 1, ==6 → 2}` 로 재라벨 → **KNN**(비모수적, 거리법이라 제약 충족)
이 신규 클래스를 게이팅. KNN 이 신규(1/2)로 판정하면 그 클래스(+4)를, 기존(0)이면 **동결 5-way 분류기**를 사용.

**모범답안2 개선**: 원본 reference 의 KNN(euclidean, k=3) 대신 **cosine 거리 + k=1** 사용 +
**전체 train-dev** 로 KNN 학습(원본은 내부 분할로 일부만 사용). → test macro-F1 0.789 → **0.793**.

신규 클래스 5·6 의 상호 혼동이 병목이며, 학습이 금지된 본 과제에선 ~0.79 가 사실상 상한입니다
(공식 reference·best 풀이 모두 동일 점수대).


## 학습 데이터셋

In [ ]:
import os, glob, numpy as np, torch
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

def _find(*cands):
    for c in cands:
        if c and os.path.exists(c): return c
    for c in cands:
        g=glob.glob(c) if c else []
        if g: return g[0]
    raise FileNotFoundError(cands)

# 로컬(Solution/ 기준) · Colab · 문제루트 모두 대응
TRAIN = _find("../training_set/train-dev_dataset_with_labels.pt",
              "training_set/train-dev_dataset_with_labels.pt", "**/train-dev_dataset_with_labels.pt")
BASE  = _find("../training_set/base_classifier.pth", "training_set/base_classifier.pth", "**/base_classifier.pth")
TEST  = _find("./test_set/test_dataset.pt", "Solution/test_set/test_dataset.pt", "**/test_set/test_dataset.pt")
TESTL = _find("./test_set/test_labels.txt", "Solution/test_set/test_labels.txt", "**/test_labels.txt")
VAL   = _find("./validation_set/eval_dataset.pt", "Solution/validation_set/eval_dataset.pt", "**/eval_dataset.pt")
print("train:", TRAIN); print("test:", TEST)

ds = torch.load(TRAIN, weights_only=False)
train_inputs = ds[:, :, :-1].reshape(-1, 768).numpy()
train_labels = ds[:, :, -1].reshape(-1).numpy().astype(int)
print("train samples:", train_inputs.shape, "classes:", sorted(set(train_labels)))

base_clf = torch.nn.Linear(768, 5)
base_clf.load_state_dict(torch.load(BASE)); base_clf.eval()


# 풀이 (Solution)

베이스라인은 새 라벨(5, 6)을 **무작위로** 배정했다. 이 셀을 아래 **모범답안2 풀이**로 교체한다 — 베이스라인의 *"You can replace the code below with your solution."* 자리에 해당한다.

**아이디어 (cosine-KNN 게이팅)**: 학습 라벨을 `기존<5 → 0 / 5 → 1 / 6 → 2` 로 재라벨해 **KNN(cosine, k=1)** 게이트를 전체 train-dev 로 학습. 추론 시 KNN 이 **신규**(1/2)면 `which+4`(=5 또는 6), **기존**(0)이면 동결 5-way `base_clf` 예측을 쓴다. (원본 reference 의 euclidean·k=3 → **cosine·k=1** 개선, macro-F1 0.789→0.793)

In [ ]:
# 재라벨: 기존<5 → 0, 클래스5 → 1, 클래스6 → 2
gate_labels = np.where(train_labels < 5, 0, np.where(train_labels == 5, 1, 2))
knn = KNeighborsClassifier(n_neighbors=1, weights='distance', metric='cosine')
knn.fit(train_inputs, gate_labels)
print("KNN(cosine, k=1) 학습 완료 — 게이트 분포:", np.bincount(gate_labels))

# 7-way 분류기 = 동결 5-way + KNN 게이팅
def predict_batch(X):
    """X: (N,768) numpy → 7-way 예측."""
    which = knn.predict(X)                      # 0=기존,1=클래스5,2=클래스6
    with torch.no_grad():
        old = torch.softmax(base_clf(torch.tensor(X, dtype=torch.float)), 1).argmax(1).numpy()
    return np.array([(w + 4) if w else int(old[i]) for i, w in enumerate(which)])

def infer(path):
    X = torch.load(path, weights_only=False).reshape(-1, 768).numpy()
    return predict_batch(X)


## 추론 및 평가

In [ ]:
gt = {}
for line in open(TESTL):
    p = line.strip().split(",")
    if len(p) >= 2 and p[0].isdigit():
        gt[int(p[0])] = int(p[1])
test_pred = infer(TEST)
y_true = [gt[i] for i in range(len(test_pred))]
f1 = f1_score(y_true, list(test_pred), average='macro')
print(f"Test macro-F1: {f1:.4f}")
print("per-class F1:", np.round(f1_score(y_true, list(test_pred), average=None), 3))


## 검증 데이터셋
검증셋 예측 → `submission_val.csv` (정답 비공개, 참고용).

In [ ]:
import pandas as pd
def submission_to_csv(pred, path):
    pred = np.array(pred).flatten()
    pd.DataFrame({"ID": np.arange(pred.size), "class": pred}).to_csv(path, index=False)

submission_to_csv(infer(VAL), "submission_val.csv")     # 검증셋 예측(정답 없음, 참고)
print("submission_val.csv 생성 완료")

## 테스트 데이터셋
테스트셋 예측 → 채점 대상 `submission.csv` (ID, class).

In [ ]:
submission_to_csv(test_pred, "submission.csv")          # 채점 대상(test 700)
print("submission.csv (test):", len(test_pred), "행 생성 완료")

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)